In [ ]:
import baostock as bs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import requests
from bs4 import BeautifulSoup
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
import warnings

# 屏蔽警告
warnings.filterwarnings('ignore')
# 解决matplotlib中文乱码
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 20只A股蓝筹代码映射
stock_map = {
    "sh.600000": "600000", "sh.600036": "600036", "sh.601318": "601318",
    "sh.601398": "601398", "sh.601988": "601988", "sh.601288": "601288",
    "sh.601668": "601668", "sz.000001": "000001", "sh.600104": "600104",
    "sh.600016": "600016", "sh.601166": "601166", "sh.601328": "601328",
    "sh.601818": "601818", "sh.601088": "601088", "sh.601601": "601601",
    "sh.601336": "601336", "sh.600519": "600519", "sh.600030": "600030",
    "sh.600019": "600019", "sh.601857": "601857"
}

# 回测时间区间、基准指数
start_date = "2025-01-01"
end_date = "2026-05-25"
benchmark_code = "sh.000300"

fin_data = []
# 循环爬取财务数据（网易页面解析失败，异常分支填充模拟数据）
for bs_code, code in stock_map.items():
    try:
        url = f"http://quotes.money.163.com/trade/{'1' if code.startswith('6') else '0'}{code}.html#01b05"
        res = requests.get(url, timeout=5)
        soup = BeautifulSoup(res.text, 'html.parser')
        # 原网页解析失效，此处仅占位，无真实数据提取逻辑
        fin_data.append({
            "ts_code": bs_code,
            "roe": 12.0,
            "bps": 6.0,
            "收入同比": 8.0,
            "利润同比": 10.0,
            "资产负债率": 60.0
        })
    except Exception as e:
        # 网页访问/解析失败，填充兜底模拟财务指标
        fin_data.append({
            "ts_code": bs_code,
            "roe": 11.0,
            "bps": 5.5,
            "收入同比": 7.0,
            "利润同比": 9.0,
            "资产负债率": 62.0
        })
    time.sleep(0.3)

# 财务数据预处理
df_fin = pd.DataFrame(fin_data)
numeric_cols = ["roe", "bps", "收入同比", "利润同比", "资产负债率"]
for col in numeric_cols:
    df_fin[col] = pd.to_numeric(df_fin[col], errors='coerce')
    df_fin[col] = df_fin[col].fillna(df_fin[col].mean())

# PCA综合打分选股
scaler = StandardScaler()
X_std = scaler.fit_transform(df_fin[numeric_cols])
pca = PCA(n_components=0.95)
Y = pca.fit_transform(X_std)
df_fin["score"] = np.sum(Y * pca.explained_variance_ratio_, axis=1)
# 筛选前12只基本面优质股票
top_stocks = df_fin.sort_values("score", ascending=False).head(12)["ts_code"].tolist()

# 登录baostock获取行情数据
bs.login()
k_list = []
for code in top_stocks:
    rs = bs.query_history_k_data_plus(
        code=code,
        fields="date,open,high,low,close,volume",
        start_date=start_date,
        end_date=end_date
    )
    df = rs.get_data().replace("", np.nan).dropna()
    df["code"] = code
    k_list.append(df)

# 获取沪深300基准行情
bench_rs = bs.query_history_k_data_plus(
    code=benchmark_code,
    fields="date,open,close",
    start_date=start_date,
    end_date=end_date
)
df_bench = bench_rs.get_data().replace("", np.nan).dropna()
bs.logout()

# 拼接全部个股行情并数值转换
k_df = pd.concat(k_list)
num_cols = ["open", "high", "low", "close", "volume"]
k_df[num_cols] = k_df[num_cols].apply(pd.to_numeric, errors="coerce")
df_bench[["open", "close"]] = df_bench[["open", "close"]].apply(pd.to_numeric, errors="coerce")

# 技术指标计算函数
def MA(c, n):
    return c.rolling(n).mean()

def MACD(c):
    e12 = c.ewm(span=12).mean()
    e26 = c.ewm(span=26).mean()
    dif = e12 - e26
    dea = dif.ewm(span=9).mean()
    return 2 * (dif - dea)

def KDJ(h, l, c):
    hh = h.rolling(9).max()
    ll = l.rolling(9).min()
    rsv = (c - ll) / (hh - ll) * 100
    k = rsv.ewm(alpha=1/3).mean()
    d = k.ewm(alpha=1/3).mean()
    j = 3 * k - 2 * d
    return k, d, j

def RSI(c, n):
    diff = c.diff()
    up = diff.mask(diff < 0, 0)
    dn = -diff.mask(diff > 0, 0)
    sum_up = up.rolling(n).sum()
    sum_dn = dn.rolling(n).sum()
    return sum_up / (sum_up + sum_dn) * 100

# 构建机器学习数据集
all_data = []
for code in top_stocks:
    d = k_df[k_df["code"] == code].sort_values("date")
    if len(d) < 60:
        continue
    c = d["close"]
    d["ma5"] = MA(c, 5)
    d["ma10"] = MA(c, 10)
    d["ma20"] = MA(c, 20)
    d["macd"] = MACD(c)
    d["k"], d["d"], d["j"] = KDJ(d["high"], d["low"], c)
    d["rsi6"] = RSI(c, 6)
    # 标签：未来3日上涨超1%为1，否则-1
    d["y"] = np.where(c.shift(-3) > c * 1.01, 1, -1)
    all_data.append(d)

model_df = pd.concat(all_data).dropna()
feats = ["ma5", "ma10", "ma20", "macd", "k", "d", "j", "rsi6"]

# 划分训练集、测试集
split = int(len(model_df) * 0.8)
train, test = model_df.iloc[:split], model_df.iloc[split:]
Xtrain = scaler.fit_transform(train[feats])
ytrain = train["y"]

# 逻辑回归模型训练
model = LogisticRegression(max_iter=1000)
model.fit(Xtrain, ytrain)
acc = model.score(Xtrain, ytrain)
test = test.reset_index(drop=True)

# 计算策略收益率
str_returns = []
for i in range(len(test) - 3):
    buy_price = test["open"].iloc[i+1]
    sell_price = test["close"].iloc[i+3]
    str_returns.append((sell_price - buy_price) / buy_price)

# 计算沪深300基准收益率
bench_returns = []
for i in range(len(df_bench) - 3):
    b_buy = df_bench["open"].iloc[i+1]
    b_sell = df_bench["close"].iloc[i+3]
    bench_returns.append((b_sell - b_buy) / b_buy)

# 收益汇总
strategy_return = sum(str_returns)
bench_return = sum(bench_returns)
excess_return = strategy_return - bench_return

# 输出指标
print(f"逻辑回归准确率:{acc:.2%}")
print(f"策略总收益率:{strategy_return:.2%}")
print(f"沪深300 基准收益率:{bench_return:.2%}")
print(f"超额收益:{excess_return:.2%}")

# 净值曲线绘图
net1 = np.cumprod(1 + np.array(str_returns))
net2 = np.cumprod(1 + np.array(bench_returns[:len(str_returns)]))
plt.figure(figsize=(12, 5))
plt.plot(net1, label="量化策略净值", linewidth=2)
plt.plot(net2, label="沪深300 基准净值", linestyle="--")
plt.axhline(1, color="gray", linestyle=":")
plt.legend()
plt.grid(alpha=0.3)
plt.show()